In [1]:
!pip install llama-index llama-index-core llama-index-readers-file pypdf -q
!pip install llama-index-embeddings-huggingface -q
!pip install llama-index-llms-gemini -q
!pip install chromadb llama-index-vector-stores-chroma -q
!pip install llama-index-retrievers-bm25 -q
!pip install sentence-transformers rank_bm25 -q
!pip install gradio -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kds 0.1.3 requires matplotlib>=1.4.0, which is not installed.
torch 2.11.0 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.


In [2]:
!pip install llama-index-llms-google-genai -q

In [3]:
import os
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI

# Embedding model
Settings.embed_model = HuggingFaceEmbedding(
    model_name="all-MiniLM-L6-v2"
)

# Gemini API key: read from the environment, or prompt for it (never hard-code keys)
from getpass import getpass
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass("Gemini API key: ")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Gemini LLM
Settings.llm = GoogleGenAI(
    model="gemini-2.5-flash",
    api_key=GOOGLE_API_KEY
)

print("Models loaded successfully.")

/opt/anaconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 17969.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully.


In [5]:
import os
import shutil

os.makedirs("docs", exist_ok=True)

filename = "NVIDIA-2025-Annual-Report.pdf"

shutil.copy(filename, f"docs/{os.path.basename(filename)}")

print(f"Moved to docs/{os.path.basename(filename)}")

Moved to docs/NVIDIA-2025-Annual-Report.pdf


In [6]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

print('Loading PDF...')
documents = SimpleDirectoryReader('docs').load_data()
print(f'Loaded {len(documents)} pages.')

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)
print(f'Split into {len(nodes)} chunks.')

print('\n--- Example Chunk ---')
print(nodes[0].get_content())

Loading PDF...
Loaded 181 pages.
Split into 450 chunks.

--- Example Chunk ---
2025
NVIDIA Corporation
Annual Review
Notice of Annual Meeting 
Proxy Statement 
Form 10-K


In [7]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

# Initialise ChromaDB — same as class notebook
print('Initialising ChromaDB...')
db = chromadb.PersistentClient(path='./chroma_db')
chroma_collection = db.get_or_create_collection('nvidia_rag')

# Connect ChromaDB to LlamaIndex
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Embed all chunks and store in ChromaDB
print('Building vector index — this will take a minute...')
index = VectorStoreIndex(nodes, storage_context=storage_context)
print('Index built successfully.')

Initialising ChromaDB...
Building vector index — this will take a minute...
Index built successfully.


In [8]:
!pip install nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()

from llama_index.core import PromptTemplate

# Vanilla prompt
VANILLA_PROMPT = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Answer the question: {query_str}\n"
)

# Baseline query engine
baseline_engine = index.as_query_engine(
    similarity_top_k=3,
    text_qa_template=VANILLA_PROMPT
)

# Test it
query = "What was NVIDIA's total revenue in 2024?"
print(f'Query: {query}\n')

response = baseline_engine.query(query)
print(f'Baseline Answer:\n{response.response}')

print('\n--- Source Chunks Used ---')
for i, node in enumerate(response.source_nodes):
    print(f'\nSource {i+1}:')
    print(node.get_content()[:200])

Query: What was NVIDIA's total revenue in 2024?



2026-06-10 22:02:16,384 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:02:19,108 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Baseline Answer:
NVIDIA's total revenue in 2024 was $60,922 million.

--- Source Chunks Used ---

Source 1:
NVIDIA Corporation and Subsidiaries
Consolidated Statements of Income
(In millions, except per share data)
Year Ended
Jan 26, 2025 Jan 28, 2024 Jan 29, 2023
Revenue $ 130,497 $ 60,922 $ 26,974 
Cost o

Source 2:
We account for 
customer programs as a reduction to revenue and accrue for such programs for potential rebates and MDFs based on the 
amount we expect to be claimed by customers.
NVIDIA Corporation an

Source 3:
Jan 26, 2025 Jan 28, 2024
Accrued and Other Current Liabilities: (In millions)
Customer program accruals $ 4,880 $ 2,081 
Excess inventory purchase obligations (1)  2,095  1,655 
Product warranty and 


In [9]:
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)
gemini_raw = genai.GenerativeModel('gemini-2.5-flash')

def rewrite_query(query: str) -> str:
    """
    Use Gemini to rewrite the query before retrieval.
    Short or vague queries embed poorly — rewriting adds
    specific financial terminology that better matches
    the language used in the annual report.
    """
    prompt = (
        'You are a search query optimizer for a financial document database.\n'
        'Rewrite the following question to improve document retrieval.\n'
        'Add relevant financial terminology and be more specific.\n'
        'Return ONLY the rewritten query, nothing else.\n\n'
        f'Original: {query}\n'
        'Rewritten:'
    )
    response = gemini_raw.generate_content(prompt)
    return response.text.strip()

# Test it
test_queries = [
    "how did nvidia do",
    "any risks?",
    "what about data centers"
]

for q in test_queries:
    rewritten = rewrite_query(q)
    print(f'Original:  {q}')
    print(f'Rewritten: {rewritten}\n')

/var/folders/hl/fd8_g75d53516f13q8kr_9rw0000gn/T/ipykernel_50949/735762651.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Original:  how did nvidia do
Rewritten: NVIDIA NVDA latest quarterly earnings report financial results revenue net income EPS stock performance

Original:  any risks?
Rewritten: Identify material financial risk factors, including market risk, credit risk, operational risk, liquidity risk, and regulatory compliance risk, detailing their potential adverse impact on financial performance or solvency and outlining risk mitigation strategies.

Original:  what about data centers
Rewritten: Financial analysis of data center infrastructure: capital expenditure, operational costs, revenue streams, profitability, and market investment trends.



In [10]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.schema import NodeWithScore
from typing import List, Dict

# Build both retrievers over the same 403 chunks
bm25_retriever   = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=10)
vector_retriever = VectorIndexRetriever(index=index, similarity_top_k=10)

def reciprocal_rank_fusion(
    bm25_results: List[NodeWithScore],
    vector_results: List[NodeWithScore],
    k: int = 60,
    top_k: int = 5
) -> List[NodeWithScore]:
    """
    Combine BM25 and vector rankings.
    Each doc scores 1/(60 + rank) per list — summed across both.
    Higher total score = more relevant by both methods.
    """
    scores: Dict[str, float] = {}
    node_map: Dict[str, NodeWithScore] = {}

    for rank, node in enumerate(bm25_results, start=1):
        nid = node.node.node_id
        scores[nid]   = scores.get(nid, 0.0) + 1.0 / (k + rank)
        node_map[nid] = node

    for rank, node in enumerate(vector_results, start=1):
        nid = node.node.node_id
        scores[nid]   = scores.get(nid, 0.0) + 1.0 / (k + rank)
        node_map[nid] = node

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [node_map[nid] for nid, _ in ranked[:top_k]]


def hybrid_retrieve(query: str, top_k: int = 5) -> List[NodeWithScore]:
    """Run BM25 + vector search and fuse with RRF."""
    bm25_results   = bm25_retriever.retrieve(query)
    vector_results = vector_retriever.retrieve(query)
    return reciprocal_rank_fusion(bm25_results, vector_results, top_k=top_k)


# Test it
test_nodes = hybrid_retrieve("NVIDIA data center revenue growth")
print(f'Hybrid retrieval returned {len(test_nodes)} chunks\n')
for i, n in enumerate(test_nodes):
    print(f'[{i+1}] {n.node.get_content()[:150]}...')
    print()

2026-06-10 22:02:57,564 - DEBUG - Building index from IDs objects


Hybrid retrieval returned 5 chunks

[1] Reportable Segments
Revenue by Reportable Segments
Year Ended
Jan 26, 2025 Jan 28, 2024
$
Change
%
Change
($ in millions)
Compute & Networking $ 116,1...

[2] Sales to direct customers which represented 10% or more of total revenue, all of which were primarily attributable to the 
Compute & Networking segmen...

[3] BUSINESS OVERVIEW
Fiscal 2025 marked an extraordinary year for NVIDIA’s growth with revenue surging 114% year on year to $130.5 billion on 
strength a...

[4] Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations 
The following discussion and analysis of our financial ...

[5] Concentration of Revenue
We refer to customers who purchase products directly from NVIDIA as direct customers, such as AIBs, distributors, 
ODMs, OEMs...



In [11]:
from sentence_transformers import CrossEncoder

print('Loading reranker...')
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Reranker loaded.')

def rerank(query: str, nodes: List[NodeWithScore], top_k: int = 3) -> List[NodeWithScore]:
    """
    Re-score each chunk against the query jointly.
    Unlike vector search which embeds query and chunk separately,
    the cross-encoder reads both together — much more accurate.
    """
    texts = [n.node.get_content() for n in nodes]
    pairs = [(query, text) for text in texts]

    scores = reranker_model.predict(pairs)

    reranked = sorted(zip(nodes, scores.tolist()), key=lambda x: x[1], reverse=True)
    return [node for node, _ in reranked[:top_k]]


# Test — rerank the 5 hybrid results down to top 3
query = "NVIDIA data center revenue growth"
reranked_nodes = rerank(query, test_nodes, top_k=3)

print(f'Top 3 after reranking:\n')
for i, n in enumerate(reranked_nodes):
    print(f'[{i+1}] {n.node.get_content()[:150]}...')
    print()

Loading reranker...


2026-06-10 22:03:01,468 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-10 22:03:01,704 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-10 22:03:01,727 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/config.json "HTTP/1.1 200 OK"
2026-06-10 22:03:01,747 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/config.json "HTTP/1.1 200 OK"
2026-06-10 22:03:01,990 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/model.safetensors "HTTP/1.1 307 Temporary Redirect"
2026-06-10 22:03:02,224 - INFO - HTTP Request: HEAD https://huggingfa

Reranker loaded.


Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00,  2.34it/s]

Top 3 after reranking:

[1] BUSINESS OVERVIEW
Fiscal 2025 marked an extraordinary year for NVIDIA’s growth with revenue surging 114% year on year to $130.5 billion on 
strength a...

[2] Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations 
The following discussion and analysis of our financial ...

[3] Reportable Segments
Revenue by Reportable Segments
Year Ended
Jan 26, 2025 Jan 28, 2024
$
Change
%
Change
($ in millions)
Compute & Networking $ 116,1...



In [12]:
GROUNDED_PROMPT = """
You are a financial analyst assistant. Answer the question using ONLY
the context provided below. Do not use any outside knowledge.
If the context does not contain enough information, say so explicitly.
Always cite specific figures and facts from the context.

Retrieved Context:
{context}

Question: {question}

Answer (grounded in context only):
"""

def enhanced_query(question: str, verbose: bool = True) -> str:
    """
    Full enhanced pipeline:
    1. Rewrite query
    2. Hybrid retrieval (BM25 + vector + RRF)
    3. Cross-encoder reranking
    4. Grounded generation
    """
    # Step 1: Query rewriting
    rewritten = rewrite_query(question)
    if verbose:
        print(f'[1] Original:  {question}')
        print(f'    Rewritten: {rewritten}\n')

    # Step 2: Hybrid retrieval
    candidates = hybrid_retrieve(rewritten, top_k=8)
    if verbose:
        print(f'[2] Retrieved {len(candidates)} candidates via hybrid search')

    # Step 3: Reranking
    top_nodes = rerank(rewritten, candidates, top_k=3)
    if verbose:
        print(f'[3] Reranked to top {len(top_nodes)} chunks\n')

    # Step 4: Grounded generation
    context  = '\n\n---\n\n'.join([n.node.get_content() for n in top_nodes])
    prompt   = GROUNDED_PROMPT.format(context=context, question=question)
    response = gemini_raw.generate_content(prompt)
    answer   = response.text.strip()

    if verbose:
        print(f'[4] Answer:\n{answer}')

    return answer

# Test the full pipeline
enhanced_query("What was NVIDIA's total revenue in 2024?")

[1] Original:  What was NVIDIA's total revenue in 2024?
    Rewritten: NVIDIA Corporation (NVDA) consolidated net revenue, fiscal year 2024 (ended January 28, 2024), Form 10-K.

[2] Retrieved 8 candidates via hybrid search


Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00,  3.71it/s]


[3] Reranked to top 3 chunks

[4] Answer:
NVIDIA's total revenue for the year ended Jan 28, 2024, was $60,922 million.


"NVIDIA's total revenue for the year ended Jan 28, 2024, was $60,922 million."

In [13]:
# Test with questions the document can actually answer
test_questions = [
    "What was NVIDIA's total revenue in 2024?",
    "What are the main risk factors NVIDIA faces?",
    "How did the Data Center segment perform?"
]

for q in test_questions:
    print('=' * 50)
    print(f'Q: {q}')
    answer = enhanced_query(q, verbose=True)
    print(f'\nFINAL ANSWER: {answer}')
    print()

Q: What was NVIDIA's total revenue in 2024?
[1] Original:  What was NVIDIA's total revenue in 2024?
    Rewritten: NVIDIA Corporation total revenue, net sales, or top-line figures for fiscal year 2024 (FY24), specifically from its annual report, 10-K filing, or consolidated income statement.

[2] Retrieved 8 candidates via hybrid search


Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00,  7.50it/s]


[3] Reranked to top 3 chunks

[4] Answer:
NVIDIA's total revenue for the year ended Jan 28, 2024, was $60,922 million.

FINAL ANSWER: NVIDIA's total revenue for the year ended Jan 28, 2024, was $60,922 million.

Q: What are the main risk factors NVIDIA faces?
[1] Original:  What are the main risk factors NVIDIA faces?
    Rewritten: Analyze **NVIDIA Corporation's principal material risk factors** affecting **financial performance** and **business operations**. Specifically, address **geopolitical export controls** on **AI chips**, **semiconductor supply chain vulnerabilities**, **competitive landscape** in **data center** and **GPU markets**, **technological obsolescence**, **regulatory compliance**, and **market demand fluctuations** for **accelerated computing solutions**.

[2] Retrieved 8 candidates via hybrid search


Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 13.55it/s]

[3] Reranked to top 3 chunks



[4] Answer:
NVIDIA faces several main risk factors:

1.  **Compliance with existing or future governmental regulations:** These regulations, including but not limited to those pertaining to IP ownership and infringement, taxes, import and export requirements and tariffs, anti-corruption, business acquisitions, foreign exchange controls and cash repatriation restrictions, data privacy requirements, competition and antitrust, advertising, employment, product regulations, cybersecurity, environmental, health and safety requirements, the responsible use of AI, climate change, cryptocurrency, and consumer laws, could "further increase our costs, impact our competitive position, and otherwise may have a material adverse impact on our business, financial condition and results of operations in subsequent periods." While compliance "has not otherwise had a material effect upon our capital expenditures, results of operations, or competitive position" previously, and they "do not currently antici

Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 32.78it/s]


[3] Reranked to top 3 chunks

[4] Answer:
The Data Center market platform experienced exceptional growth and was a key contributor to NVIDIA's performance in fiscal 2025.

Here's how it performed:
*   **Revenue:** It generated $115.2 billion in revenue in fiscal 2025.
*   **Year-over-Year Growth:** Revenue for Data Center increased by 142% year on year.
*   **Drivers of Growth:** This growth was led by exceptional demand for the Hopper architecture, which is used for large language models, recommendation engines, and generative AI applications. Ethernet for AI was another key contributor, including strong uptake of the Spectrum-X end-to-end ethernet platform.
*   **Segment Contribution:** Within the Compute & Networking segment, Data Center computing revenue grew 162% driven primarily by demand for the Hopper computing platform. Revenue from Data Center networking grew 51% driven by Ethernet for AI revenue.
*   **Impact on Gross Margin:** A higher mix of Data Center revenue was the pri

In [14]:
# Test query set — four categories: simple, deep-context, ambiguous, edge case
TEST_QUERIES = [
    # Simple factual
    {"query": "What was NVIDIA's total revenue in 2024?",        "type": "simple_fact"},
    {"query": "What was the gross profit margin in 2024?",       "type": "simple_fact"},
    # Deep context
    {"query": "What are the main risk factors NVIDIA faces?",    "type": "deep_context"},
    {"query": "How did the Data Center segment perform?",        "type": "deep_context"},
    # Ambiguous
    {"query": "How did NVIDIA do recently?",                     "type": "ambiguous"},
    {"query": "Is NVIDIA a good investment?",                    "type": "ambiguous"},
    # Edge cases
    {"query": "What were NVIDIA's revenues in 2045?",           "type": "edge_case"},
    {"query": "Tell me about NVIDIA's ice cream division",      "type": "edge_case"},
]

print('Running evaluation — baseline vs enhanced...\n')

results = []

for item in TEST_QUERIES:
    q    = item["query"]
    qtype = item["type"]
    print(f'[{qtype}] {q}')

    # Baseline
    base_resp   = baseline_engine.query(q)
    base_answer = base_resp.response

    # Enhanced
    enh_answer = enhanced_query(q, verbose=False)

    # Score both using Gemini as judge
    score_prompt = f"""Score these two answers to the question below.
Rate each on 3 dimensions from 1-5:
- Correctness: Is it factually accurate based on the context?
- Grounding: Does it cite specific facts rather than generalise?
- Completeness: Does it fully answer the question?

Question: {q}
Answer A (Baseline): {base_answer}
Answer B (Enhanced): {enh_answer}

Reply ONLY in this format:
BASELINE_CORRECTNESS: <1-5>
BASELINE_GROUNDING: <1-5>
BASELINE_COMPLETENESS: <1-5>
ENHANCED_CORRECTNESS: <1-5>
ENHANCED_GROUNDING: <1-5>
ENHANCED_COMPLETENESS: <1-5>"""

    score_resp = gemini_raw.generate_content(score_prompt)
    scores = {}
    for line in score_resp.text.strip().split('\n'):
        if ':' in line:
            key, val = line.split(':', 1)
            try:
                scores[key.strip()] = int(val.strip())
            except:
                scores[key.strip()] = 0

    results.append({
        "query": q,
        "type": qtype,
        "baseline_answer": base_answer,
        "enhanced_answer": enh_answer,
        "scores": scores
    })
    print(f'  Baseline: C={scores.get("BASELINE_CORRECTNESS",0)} G={scores.get("BASELINE_GROUNDING",0)} Co={scores.get("BASELINE_COMPLETENESS",0)}')
    print(f'  Enhanced: C={scores.get("ENHANCED_CORRECTNESS",0)} G={scores.get("ENHANCED_GROUNDING",0)} Co={scores.get("ENHANCED_COMPLETENESS",0)}\n')

# Summary
import numpy as np
dims = [("Correctness","_CORRECTNESS"), ("Grounding","_GROUNDING"), ("Completeness","_COMPLETENESS")]
print('='*50)
print(f'{"Dimension":<15} {"Baseline":>10} {"Enhanced":>10}')
print('-'*35)
for label, key in dims:
    b = np.mean([r["scores"].get("BASELINE"+key, 0) for r in results])
    e = np.mean([r["scores"].get("ENHANCED"+key, 0) for r in results])
    print(f'{label:<15} {b:>10.2f} {e:>10.2f}')

2026-06-10 22:04:46,747 - INFO - AFC is enabled with max remote calls: 10.


Running evaluation — baseline vs enhanced...

[simple_fact] What was NVIDIA's total revenue in 2024?


2026-06-10 22:04:49,353 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 35.01it/s]


  Baseline: C=2 G=5 Co=2
  Enhanced: C=5 G=5 Co=5

[simple_fact] What was the gross profit margin in 2024?


2026-06-10 22:05:14,420 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:05:16,667 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00,  8.76it/s]


...

  Baseline: C=5 G=4 Co=5
  Enhanced: C=5 G=5 Co=3

[edge_case] What were NVIDIA's revenues in 2045?


2026-06-10 22:07:56,105 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:07:58,840 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 29.40it/s]
2026-06-10 22:08:10,888 - INFO - AFC is enabled with max remote calls: 10.


  Baseline: C=5 G=5 Co=5
  Enhanced: C=5 G=5 Co=5

[edge_case] Tell me about NVIDIA's ice cream division


2026-06-10 22:08:13,723 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 41.32it/s]


  Baseline: C=5 G=4 Co=5
  Enhanced: C=5 G=5 Co=5

Dimension         Baseline   Enhanced
-----------------------------------
Correctness           4.12       4.50
Grounding             3.75       4.75
Completeness          3.88       4.62


In [15]:
DEMO_QUERIES = [
    "What was NVIDIA's total revenue in 2024?",                    # simple fact
    "What are the main risk factors NVIDIA faces?",                # deep context
    "Tell me about NVIDIA's ice cream division",                   # edge case
]

print('=' * 60)
print('DEMO LOG — Baseline vs Enhanced RAG')
print('NVIDIA 2024 Annual Report')
print('=' * 60)

for q in DEMO_QUERIES:
    print(f'\nQUERY: {q}')
    print('-' * 40)

    # Baseline
    base_resp = baseline_engine.query(q)
    print(f'BASELINE:\n{base_resp.response}')
    print()

    # Enhanced
    print('ENHANCED:')
    enh_answer = enhanced_query(q, verbose=True)
    print(f'\nFINAL: {enh_answer}')
    print('=' * 60)

2026-06-10 22:08:31,577 - INFO - AFC is enabled with max remote calls: 10.


DEMO LOG — Baseline vs Enhanced RAG
NVIDIA 2024 Annual Report

QUERY: What was NVIDIA's total revenue in 2024?
----------------------------------------


2026-06-10 22:08:34,207 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


BASELINE:
NVIDIA's total revenue in 2024 was **$60,922 million**.

ENHANCED:
[1] Original:  What was NVIDIA's total revenue in 2024?
    Rewritten: NVIDIA (NVDA) consolidated net revenue for fiscal year 2024 as reported in 10-K filing or annual earnings report.

[2] Retrieved 8 candidates via hybrid search


Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 25.81it/s]


...

[4] Answer:
NVIDIA faces several main risk factors, as detailed in the provided context:

**From the "Risk Factors Summary" section:**

*   **Risks Related to Our Industry and Markets**:
    *   Failure to meet the evolving needs of our industry and markets may adversely impact our financial results.
    *   Competition could adversely impact our market share and financial results.
*   **Risks Related to Demand, Supply, and Manufacturing**:
    *   Long manufacturing lead times and uncertain supply and component availability, combined with a failure to estimate customer demand accurately has led and could lead to mismatches between supply and demand.
    *   Dependency on third-party suppliers and their technology to manufacture, assemble, test, or package our products reduces our control over product quantity and quality, manufacturing yields, and product delivery schedules and could harm our business.
    *   Defects in our products have caused and could cause us to incur significant

2026-06-10 22:09:18,145 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


BASELINE:
Based on the provided context, there is no information about an "ice cream division" for NVIDIA. The context describes NVIDIA as a company focused on accelerated computing, GPUs, AI, and semiconductors, and details about its CEO, Jen-Hsun Huang, and competitive landscape in the tech industry.

ENHANCED:
[1] Original:  Tell me about NVIDIA's ice cream division
    Rewritten: NVIDIA Corporation segment reporting, business diversification strategy, and financial performance for any consumer goods, food and beverage, or non-core operating divisions or subsidiaries.

[2] Retrieved 8 candidates via hybrid search


Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 33.27it/s]


[3] Reranked to top 3 chunks

[4] Answer:
The provided context describes NVIDIA's reportable segments as "Compute & Networking," "Graphics," and an "All Other" category. It details the revenue and operating income for these segments for the years ended January 26, 2025, January 28, 2024, and January 29, 2023.

The context does not contain any information about an "ice cream division" for NVIDIA. Therefore, I cannot provide details about it.

FINAL: The provided context describes NVIDIA's reportable segments as "Compute & Networking," "Graphics," and an "All Other" category. It details the revenue and operating income for these segments for the years ended January 26, 2025, January 28, 2024, and January 29, 2023.

The context does not contain any information about an "ice cream division" for NVIDIA. Therefore, I cannot provide details about it.


In [17]:
# Retrieval Precision Evaluation
from llama_index.core.retrievers import VectorIndexRetriever

def assess_relevance(query: str, chunk: str) -> int:
    """Use Gemini to judge if a chunk is relevant to the query. Returns 1 or 0."""
    prompt = (
        f'Is the following text chunk relevant to answering this question?\n'
        f'Question: {query}\n'
        f'Chunk: {chunk[:400]}\n'
        f'Reply with only YES or NO.'
    )
    response = Settings.llm.complete(prompt)
    return 1 if 'YES' in str(response).upper() else 0


eval_queries = [
    "What was NVIDIA's total revenue in 2024?",
    "What are the main risk factors NVIDIA faces?",
    "How did the Data Center segment perform?",
    "What was the gross profit margin in 2024?",
]

baseline_precisions = []
enhanced_precisions = []

print("Retrieval Precision3 Evaluation\n")
print(f"{'Query':<45} {'Baseline':>10} {'Enhanced':>10}")
print("-" * 67)

for q in eval_queries:
    # Baseline retrieval
    base_retriever = VectorIndexRetriever(index=index, similarity_top_k=3)
    base_nodes = base_retriever.retrieve(q)
    base_relevant = sum(assess_relevance(q, n.node.get_content()) for n in base_nodes)
    base_p = base_relevant / len(base_nodes)

    # Enhanced retrieval
    rewritten = rewrite_query(q)
    hybrid_nodes = hybrid_retrieve(rewritten, top_k=8)
    enh_nodes = rerank(rewritten, hybrid_nodes, top_k=3)
    enh_relevant = sum(assess_relevance(q, n.node.get_content()) for n in enh_nodes)
    enh_p = enh_relevant / len(enh_nodes)

    baseline_precisions.append(base_p)
    enhanced_precisions.append(enh_p)

    print(f"{q:<45} {base_p:>10.2f} {enh_p:>10.2f}")

print("-" * 67)
print(f"{'Average Precision3':<45} {sum(baseline_precisions)/len(baseline_precisions):>10.2f} {sum(enhanced_precisions)/len(enhanced_precisions):>10.2f}")

2026-06-10 22:12:32,279 - INFO - AFC is enabled with max remote calls: 10.


Retrieval Precision3 Evaluation

Query                                           Baseline   Enhanced
-------------------------------------------------------------------


2026-06-10 22:12:35,951 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:12:35,957 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:12:39,144 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:12:39,151 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:12:42,111 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 38.97it/s]
2026-06-10 22:12:51,974 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:12:56,043 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:12:56,048 - INFO - AFC is enabled with max r

What was NVIDIA's total revenue in 2024?            0.33       0.33


2026-06-10 22:13:04,854 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:13:04,860 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:13:07,307 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:13:07,313 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:13:10,279 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 33.09it/s]
2026-06-10 22:13:17,889 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:13:20,992 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:13:21,000 - INFO - AFC is enabled with max r

What are the main risk factors NVIDIA faces?        0.67       1.00


2026-06-10 22:13:30,414 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:13:30,420 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:13:33,383 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:13:33,389 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:13:36,386 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 46.72it/s]
2026-06-10 22:13:42,395 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:13:45,409 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:13:45,413 - INFO - AFC is enabled with max r

How did the Data Center segment perform?            0.33       0.33


2026-06-10 22:13:54,009 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:13:54,015 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:13:57,206 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:13:57,214 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:13:59,338 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 26.18it/s]
2026-06-10 22:14:05,356 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:14:09,053 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-06-10 22:14:09,060 - INFO - AFC is enabled with max r

What was the gross profit margin in 2024?           0.33       0.33
-------------------------------------------------------------------
Average Precision3                                  0.42       0.50


In [28]:
import nest_asyncio
nest_asyncio.apply()

query = input("Enter your query: ")
mode  = input("baseline or enhanced? ").strip().lower()

print(f"\nQuery: {query}")
print("=" * 60)

if mode == "baseline":
    print("\n[BASELINE]")
    print("-" * 40)
    response = baseline_engine.query(query)
    print(response.response)

else:
    print("\n[ENHANCED]")
    print("-" * 40)
    rewritten = rewrite_query(query)
    print(f"Rewritten: {rewritten}\n")
    candidates = hybrid_retrieve(rewritten, top_k=8)
    top_nodes  = rerank(rewritten, candidates, top_k=3)
    context    = '\n\n---\n\n'.join([n.node.get_content() for n in top_nodes])
    prompt     = GROUNDED_PROMPT.format(context=context, question=query)
    response   = Settings.llm.complete(prompt)
    print(str(response))

Enter your query:  Where did nvidia make mOney
baseline or enhanced?  enhanced



Query: Where did nvidia make mOney

[ENHANCED]
----------------------------------------
Rewritten: NVIDIA revenue breakdown by business segment, product line, and geographic region.



Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 77.73it/s]
2026-06-10 22:25:53,690 - INFO - AFC is enabled with max remote calls: 10.
2026-06-10 22:26:00,565 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


NVIDIA made money from various sources, totaling $130,497 million in revenue for the year ended January 26, 2025.

Geographically, based upon customer billing location for the year ended January 26, 2025, NVIDIA's revenue was:
*   **United States**: $61,257 million. The increase in revenue to the United States for fiscal years 2025 and 2024 was primarily due to higher U.S.-based Compute & Networking segment demand.
*   **Singapore**: $23,684 million, representing 18% of fiscal year 2025 total revenue. However, customers use Singapore to centralize invoicing, and shipments to Singapore were less than 2% of fiscal year 2025 total revenue.
*   **Taiwan**: $20,573 million.
*   **China (including Hong Kong)**: $17,108 million.
*   **Other**: $7,875 million.

Revenue from sales to customers outside of the United States accounted for 53% of total revenue for fiscal year 2025.

NVIDIA also made money from direct customers, which include AIBs, distributors, ODMs, OEMs, and system integrators. F